[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gustavoespi27/comida-chilena-cnn/blob/main/NB03_Preparacion_Dataset_Comida_Chilena.ipynb)
# NB03 — Preparación y gestión del dataset de imágenes: comida chilena
**Dataset del proyecto:** fotografías de 8 platos típicos chilenos (`cazuela`, `chacarero`, `completo`, `empanada`, `humitas`, `mote_con_huesillo`, `pastel_de_choclo`, `sopaipilla`).

Adaptación de la guía *NB03 — Preparación y gestión del dataset de imágenes* (Taller Fundamentos Data Science Python, Machine Learning 2026), que originalmente usa TensorFlow Flower Photos. Se aplica el mismo procedimiento al dataset propio.


## Introducción — La preparación de imágenes es el primer paso antes de construir una CNN

Una **red neuronal convolucional (CNN)** es un modelo de *Deep Learning* capaz de aprender patrones visuales para tareas como la clasificación de imágenes. Sin embargo, **no comenzamos construyendo la red**: primero necesitamos conocer, revisar y preparar las imágenes con las que aprenderá y con las que evaluaremos su desempeño.

En este notebook desarrollaremos esa **primera fase del flujo de trabajo**. Usaremos un **dataset propio de comida chilena**, organizado en ocho categorías: `cazuela`, `chacarero`, `completo`, `empanada`, `humitas`, `mote_con_huesillo`, `pastel_de_choclo`, `sopaipilla`. Las imágenes se obtuvieron mediante búsquedas web, se filtraron automáticamente (duplicados y tamaño mínimo) y se revisaron manualmente para eliminar fotografías que no correspondían a cada plato.

**Al finalizar este notebook:**
- Habremos explorado la cantidad, distribución, contenido y características técnicas de las imágenes.
- Habremos identificado archivos problemáticos y preparado etiquetas consistentes.
- Tendremos conjuntos separados de **entrenamiento (train)**, **validación (validation)** y **prueba (test)**.
- Dispondremos de imágenes redimensionadas y normalizadas, con **Data Augmentation solo para entrenamiento**, listas para alimentar una futura CNN.

> **Importante:** en el NB03 **no se construye ni se entrena una CNN**. Este notebook prepara sus datos de entrada. La construcción del modelo corresponde a una etapa posterior.

**Pregunta orientadora:** ¿qué tan confiable sería la evaluación de una CNN si no conocemos la calidad, las etiquetas o el origen de sus imágenes?

## Guía general — Ocho etapas para preparar imágenes antes de entrenar una CNN

Aunque el NB03 está organizado en 18 secciones técnicas, **todas responden a ocho etapas metodológicas**. Utilice esta ruta como referencia cuando trabaje con otro dataset: la secuencia puede adaptarse al problema, pero no deberían omitirse los controles esenciales.

| Etapa | ¿Qué hacemos? | ¿Para qué sirve? | Secciones relacionadas |
|---|---|---|---|
| **1. Obtener y organizar** | Identificar la fuente, descargar las imágenes y estructurar las carpetas por clase. | Tener datos accesibles y una organización comprensible. | 1–3 |
| **2. Explorar** | Contar archivos, observar ejemplos, revisar dimensiones y balance. | Entender qué datos tenemos realmente. | 4–7 |
| **3. Comprobar la calidad** | Detectar imágenes ilegibles, errores y posibles duplicados. | Evitar fallos de procesamiento y resultados engañosos. | 8; inspección visual en 6 |
| **4. Definir etiquetas** | Asociar cada imagen con su clase y un índice numérico consistente. | Establecer qué debe aprender a predecir la CNN. | 9 y 15 |
| **5. Separar los datos** | Crear conjuntos independientes de entrenamiento, validación y prueba. | Medir generalización sin utilizar test para ajustar decisiones. | 9 y 12 |
| **6. Estandarizar** | Redimensionar imágenes, asegurar canales compatibles y escalar valores de píxeles. | Entregar tensores adecuados y consistentes al modelo. | 9–11 y 14 |
| **7. Aumentar datos de entrenamiento** | Aplicar variaciones justificadas, como reflejos, rotaciones o zoom. | Favorecer la generalización sin alterar validación ni test. | 13–14 |
| **8. Construir y comprobar el pipeline** | Integrar lectura, transformaciones, lotes y documentación. | Dejar los conjuntos listos para la CNN y reproducir el procedimiento. | 14–16 |

**Ruta resumida:**

`Imágenes originales → organización → exploración → control de calidad → etiquetas → división train/validation/test → estandarización → aumento solo en train → pipeline listo para CNN`

**Tres precauciones que deben recordarse:**
1. **No mezclar subconjuntos.** Una imagen no debe aparecer simultáneamente en train, validation y test. Si existen duplicados o fotografías relacionadas del mismo objeto o sujeto, deben revisarse antes de dividir los datos para evitar fuga de información.
2. **No aumentar validation ni test.** Esas particiones deben conservar criterios de evaluación consistentes; solo reciben transformaciones deterministas necesarias, como redimensionamiento y normalización.
3. **No confundir demostración con automatización universal.** El NB03 detecta imágenes ilegibles, pero **no implementa una detección automática de duplicados ni una auditoría completa de etiquetas**. En un dataset propio esos controles pueden requerir trabajo adicional.

### Cómo trabajar este notebook

Siga las celdas **de arriba hacia abajo** y examine sus resultados antes de continuar. Para cada etapa, responda: *¿qué se hizo?, ¿por qué se hizo? y ¿cómo comprobamos que funcionó?* No es necesario memorizar todas las instrucciones de Python: lo importante es comprender la lógica de preparación que se reutilizará al construir una CNN.

## 1. Preparación del entorno

**Objetivo.** Importar las bibliotecas que utilizaremos durante el laboratorio. `pathlib` permite gestionar rutas; `pandas` resume datos en tablas; `matplotlib` genera gráficos; `Pillow` inspecciona imágenes y `TensorFlow/Keras` construye el pipeline para aprendizaje profundo.

**Qué verificar:** la celda debe mostrar la versión instalada de TensorFlow. Si falla una importación, revise el entorno de ejecución antes de continuar.


In [ ]:
import os
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image

print("TensorFlow:", tf.__version__)


## 2. Fuente de datos: repositorio de GitHub

**Objetivo.** Obtener el dataset del proyecto. Las imágenes de comida chilena están publicadas en el repositorio [`gustavoespi27/comida-chilena-cnn`](https://github.com/gustavoespi27/comida-chilena-cnn), junto con este notebook. **No se requiere Google Drive.**

**Qué hace el código:** si el notebook se ejecuta dentro del repositorio (por ejemplo, en un PC donde se clonó), usa esa carpeta; en Colab, descarga el repositorio con `git clone` (solo la primera vez). Los archivos descargados en Colab son temporales y desaparecen al terminar la sesión.

**Qué verificar:** la celda debe mostrar la ruta de `dataset_comida_chilena`.


In [ ]:
# Obtener el dataset desde GitHub
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/gustavoespi27/comida-chilena-cnn.git"

if Path("dataset_comida_chilena").is_dir():
    BASE_DIR = Path(".")                     # el notebook ya está dentro del repositorio
else:
    BASE_DIR = Path("comida-chilena-cnn")
    if not BASE_DIR.is_dir():                # clonar solo la primera vez
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(BASE_DIR)], check=True)

DATASET_DIR = BASE_DIR / "dataset_comida_chilena"
if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f"No se encontró el dataset: {DATASET_DIR}")
print("Dataset:", DATASET_DIR.resolve())


## 3. Ubicar el dataset

**Objetivo.** Comprobar el conjunto de imágenes etiquetadas mediante su estructura de carpetas. Cada subcarpeta representa una clase; por ejemplo, `empanada` contiene fotografías etiquetadas como empanadas.

**Qué hace el código:** lista las carpetas de clase que contiene `DATASET_DIR`.

**Qué verificar:** `DATASET_DIR` debe contener las ocho carpetas indicadas. **Pregunta:** ¿qué ocurriría si mezcláramos imágenes de dos clases en la misma carpeta?

```text
dataset_comida_chilena/
├── cazuela/
├── chacarero/
├── completo/
├── empanada/
├── humitas/
├── mote_con_huesillo/
├── pastel_de_choclo/
└── sopaipilla/
```

El archivo con el origen de cada imagen (`fuentes_dataset_comida_chilena.csv`) se guarda **fuera** de esta carpeta, para que no se cuente como imagen.


In [ ]:
carpetas_clase = sorted(p.name for p in DATASET_DIR.iterdir() if p.is_dir())
print("Carpetas de clase:", len(carpetas_clase))
for nombre in carpetas_clase:
    print(" -", nombre)


## 4. Identificar clases y cantidad de imágenes

**Objetivo.** Contar imágenes por categoría para comprender la composición del dataset. Se consideran extensiones habituales (`jpg`, `jpeg`, `png`, `bmp`) y se ignoran archivos ajenos a las imágenes.

**Qué verificar:** la tabla debe presentar una fila por clase y el total de fotografías. Estas cantidades serán la referencia para estudiar el balance.


In [ ]:
extensiones = {".jpg", ".jpeg", ".png", ".bmp"}
clases = sorted(carpeta.name for carpeta in DATASET_DIR.iterdir() if carpeta.is_dir())
if len(clases) < 2:
    raise ValueError("Se requieren al menos dos carpetas de clases.")
resumen = []
for clase in clases:
    archivos = [f for f in (DATASET_DIR / clase).iterdir() if f.is_file() and f.suffix.lower() in extensiones]
    resumen.append({"Clase": clase, "Cantidad": len(archivos)})
df_resumen = pd.DataFrame(resumen)
print("Total de imágenes:", df_resumen["Cantidad"].sum())
display(df_resumen)


## 5. Analizar el balance de clases

**Objetivo.** Detectar si algunas clases tienen muchas más imágenes que otras. Este fenómeno se denomina *desbalance de clases* y puede llevar a que el modelo aprenda mejor las categorías más frecuentes.

**Qué verificar:** compare visualmente las alturas de las barras. **Pregunta:** ¿una accuracy elevada garantiza que el modelo reconoce bien todos los platos?


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(df_resumen["Clase"], df_resumen["Cantidad"])
plt.xlabel("Clase")
plt.ylabel("Cantidad de imágenes")
plt.title("Distribución de imágenes por clase")
plt.xticks(rotation=45)
plt.show()


## 6. Visualizar ejemplos del dataset

**Objetivo.** Inspeccionar ejemplos reales antes de diseñar un modelo. La revisión visual ayuda a detectar imágenes ambiguas, fondos muy variados, posibles errores de etiquetado y diferencias entre clases.

**Qué hace el código:** selecciona aleatoriamente hasta tres imágenes de cada clase y las presenta en una cuadrícula.

**Qué verificar:** que las fotografías correspondan a las etiquetas de su carpeta. **Pregunta:** ¿qué características visuales podrían confundir al clasificador?


In [ ]:
fig, axes = plt.subplots(
    len(clases),
    3,
    figsize=(9, max(3, len(clases) * 3))
)

if len(clases) == 1:
    axes = np.array([axes])

for i, clase in enumerate(clases):
    archivos = [
        f for f in (DATASET_DIR / clase).iterdir()
        if f.suffix.lower() in extensiones
    ]

    seleccion = random.sample(
        archivos,
        min(3, len(archivos))
    )

    for j in range(3):
        ax = axes[i, j]
        ax.axis("off")

        if j < len(seleccion):
            imagen = Image.open(seleccion[j]).convert("RGB")
            ax.imshow(imagen)
            ax.set_title(clase)

plt.tight_layout()
plt.show()


## 7. Revisar dimensiones y formatos

**Objetivo.** Caracterizar la resolución, el formato y los canales de las imágenes. `ancho` y `alto` describen dimensiones en píxeles; `formato` identifica el tipo de archivo y `modo` representa su codificación de color (por ejemplo, RGB).

**Qué hace el código:** abre cada archivo legible, registra sus propiedades y calcula estadísticas descriptivas de tamaño.

**Qué verificar:** si todas las imágenes tienen la misma resolución. **Pregunta:** ¿por qué es necesario redimensionarlas antes de agruparlas en batches?


In [ ]:
registros = []
for clase in clases:
    for archivo in (DATASET_DIR / clase).iterdir():
        if archivo.suffix.lower() not in extensiones:
            continue
        try:
            with Image.open(archivo) as img:
                registros.append({"archivo": str(archivo), "clase": clase, "ancho": img.width, "alto": img.height, "formato": img.format, "modo": img.mode})
        except Exception as e:
            print("No se pudo leer:", archivo.name, e)
df_imagenes = pd.DataFrame(registros)
display(df_imagenes.head())


In [ ]:
df_imagenes[["ancho", "alto"]].describe()


## 8. Detectar archivos problemáticos

**Objetivo.** Identificar archivos corruptos o ilegibles antes de entrenar. Una imagen dañada puede interrumpir la carga del dataset y hacer fallar el entrenamiento.

**Qué hace el código:** usa `Image.verify()` para detectar errores de lectura y registra la ruta y el mensaje de error. En la sección 9 estos archivos se excluyen de las particiones.

**Qué verificar:** el número de archivos problemáticos. Si es mayor que cero, revise la lista y documente las exclusiones; el notebook no borra archivos originales.

Antes del entrenamiento conviene identificar imágenes dañadas o que no puedan abrirse correctamente.


In [ ]:
archivos_problematicos = []

for clase in clases:
    for archivo in (DATASET_DIR / clase).iterdir():
        if archivo.suffix.lower() not in extensiones:
            continue

        try:
            with Image.open(archivo) as img:
                img.verify()
        except Exception as e:
            archivos_problematicos.append({
                "archivo": str(archivo),
                "error": str(e)
            })

df_problematicos = pd.DataFrame(archivos_problematicos)

print("Archivos problemáticos:", len(df_problematicos))
df_problematicos.head()


## 9. Preparar el dataset con TensorFlow

**Objetivo.** Convertir los archivos en datasets de TensorFlow reproducibles. Primero se asigna un índice numérico a cada clase y se excluyen los archivos problemáticos. Después se realiza una división **estratificada 70/15/15**, que procura conservar la proporción de clases en cada subconjunto.

**Qué hace el código:** `train_test_split` separa rutas y etiquetas; `tf.data` lee, decodifica y redimensiona las imágenes a 224 × 224; `batch(32)` agrupa ejemplos y `prefetch` anticipa la lectura.

**Qué verificar:** cantidades de train, validation y test; la suma debe coincidir con las imágenes válidas. **Pregunta:** ¿por qué el conjunto test debe permanecer fuera de las decisiones de ajuste?

Se construyen tres subconjuntos **independientes y estratificados**: entrenamiento (70 %), validación (15 %) y test (15 %). La semilla permite reproducir la partición. Primero se separa test y luego validación; el test queda reservado para la evaluación final.

Se excluyen los archivos identificados como problemáticos. Las etiquetas se derivan de los nombres de las carpetas y mantienen un orden alfabético consistente.


In [ ]:
from sklearn.model_selection import train_test_split

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
class_names = clases
clase_a_indice = {nombre: i for i, nombre in enumerate(class_names)}
problematicos = {item["archivo"] for item in archivos_problematicos}
archivos_validos = [(str(p), clase_a_indice[clase]) for clase in clases for p in sorted((DATASET_DIR / clase).iterdir()) if p.is_file() and p.suffix.lower() in extensiones and str(p) not in problematicos]
rutas = [p for p, _ in archivos_validos]
etiquetas = [y for _, y in archivos_validos]
if min(np.bincount(etiquetas)) < 3:
    raise ValueError("Cada clase debe contar con suficientes imágenes para la división estratificada.")
X_train, X_temp, y_train, y_temp = train_test_split(rutas, etiquetas, test_size=0.30, random_state=SEED, stratify=etiquetas)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp)

def cargar_imagen(ruta, etiqueta):
    contenido = tf.io.read_file(ruta)
    imagen = tf.io.decode_image(contenido, channels=3, expand_animations=False)
    imagen.set_shape([None, None, 3])
    imagen = tf.image.resize(imagen, IMG_SIZE)
    return imagen, etiqueta

def crear_dataset(rutas, etiquetas, entrenar=False):
    ds = tf.data.Dataset.from_tensor_slices((rutas, etiquetas))
    if entrenar:
        ds = ds.shuffle(len(rutas), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(cargar_imagen, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = crear_dataset(X_train, y_train, entrenar=True)
val_ds = crear_dataset(X_val, y_val)
test_ds = crear_dataset(X_test, y_test)
print("Clases:", class_names)
print("Train:", len(X_train), "Validation:", len(X_val), "Test:", len(X_test))


## 10. Inspeccionar un batch

**Objetivo.** Comprender la estructura de los tensores que recibe una CNN. Un *batch* es un grupo de imágenes procesadas conjuntamente.

**Qué verificar:** `imagenes.shape` tiene la forma `(cantidad_del_batch, 224, 224, 3)`; el último valor representa los canales RGB. `etiquetas.shape` contiene un índice de clase por imagen. El último batch puede tener menos de 32 ejemplos.


In [ ]:
for imagenes, etiquetas in train_ds.take(1):
    print("Forma de las imágenes:", imagenes.shape)
    print("Forma de las etiquetas:", etiquetas.shape)
    print("Rango original:", float(tf.reduce_min(imagenes)), "a", float(tf.reduce_max(imagenes)))


## 11. Normalización

**Objetivo.** Escalar las intensidades de píxel. Las imágenes RGB suelen representarse con valores de 0 a 255; dividir por 255 produce valores entre 0 y 1, lo que facilita trabajar con redes neuronales.

**Qué hace el código:** `layers.Rescaling(1./255)` transforma los píxeles sin cambiar las etiquetas. En esta sección se demuestra la operación; en la sección 14 se incorpora al pipeline final.

**Qué verificar:** el rango impreso después de la transformación. **Pregunta:** ¿qué ocurriría si normalizamos solo las imágenes de entrenamiento y no las de validación?

Transformaremos los valores de los píxeles desde el rango 0–255 al rango 0–1.


In [ ]:
normalization_layer = layers.Rescaling(1./255)

train_norm = train_ds.map(
    lambda x, y: (normalization_layer(x), y)
)

val_norm = val_ds.map(
    lambda x, y: (normalization_layer(x), y)
)

for imagenes, etiquetas in train_norm.take(1):
    print(
        "Rango normalizado:",
        float(tf.reduce_min(imagenes)),
        "a",
        float(tf.reduce_max(imagenes))
    )


## 12. División Train / Validation / Test

**Objetivo.** Separar las funciones de los datos: **train** sirve para aprender los parámetros; **validation** para comparar configuraciones y decidir ajustes; **test** para la evaluación final con ejemplos reservados.

**Qué hace el notebook:** la partición ya se construyó en la sección 9. La estratificación conserva aproximadamente las proporciones por clase y la semilla 42 permite repetir la división.

**Qué verificar:** que ninguna ruta esté repetida entre subconjuntos. **Pregunta:** ¿qué es la *fuga de datos* y por qué puede producir métricas engañosas? Si existen fotografías duplicadas o muy similares, es necesario revisar también ese riesgo.

La división **70/15/15**, estratificada y reproducible, ya se realizó en la sección 9. El conjunto de entrenamiento permite ajustar parámetros; validación sirve para seleccionar y ajustar el modelo; **test no debe consultarse para tomar decisiones de entrenamiento**.

Los objetos `train_ds`, `val_ds` y `test_ds` están disponibles en memoria. Para un proyecto propio, documente la proporción utilizada, la distribución por clase y cualquier criterio especial de separación (por ejemplo, evitar imágenes duplicadas o del mismo sujeto entre subconjuntos).


## 13. Data Augmentation

**Objetivo.** Introducir variaciones realistas en las imágenes de entrenamiento para que la CNN no dependa excesivamente de orientaciones o encuadres específicos.

**Qué hace el código:** aplica reflejos horizontales, pequeñas rotaciones y zoom aleatorio, y muestra nueve variantes de una fotografía.

**Qué verificar:** que el plato siga siendo reconocible y conserve su etiqueta. **Pregunta:** ¿qué transformaciones podrían ser inapropiadas en un dataset médico o de señales de tránsito?

El aumento de datos permite generar variaciones de las imágenes de entrenamiento sin modificar las etiquetas.


In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10)
])

for imagenes, etiquetas in train_ds.take(1):
    imagen = imagenes[0]

    plt.figure(figsize=(8, 8))

    for i in range(9):
        imagen_aumentada = data_augmentation(
            tf.expand_dims(imagen, 0),
            training=True
        )

        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(
            tf.cast(imagen_aumentada[0], tf.uint8)
        )
        plt.axis("off")

    plt.tight_layout()
    plt.show()


## 14. Incorporar Data Augmentation al pipeline

**Objetivo.** Construir un flujo coherente de preparación. Las transformaciones aleatorias se aplican **solo a entrenamiento**; la normalización se aplica a los tres subconjuntos.

**Qué hace el código:** `map` transforma cada batch; `num_parallel_calls` y `prefetch` mejoran la eficiencia de procesamiento. Se crean `train_aug`, `val_preparado` y `test_preparado`.

**Qué verificar:** que todos entreguen tensores con intensidades 0–1 y que validación/test no reciban aumentos aleatorios. **Pregunta:** ¿por qué aplicar Data Augmentation al test impediría una comparación estable?


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
# Solo el entrenamiento recibe transformaciones aleatorias.
# La normalización 0–1 se aplica a los tres subconjuntos.
train_aug = train_ds.map(lambda x, y: (normalization_layer(data_augmentation(x, training=True)), y), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_preparado = val_ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_preparado = test_ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
print("Pipelines listos: train_aug, val_preparado, test_preparado")


## 15. Verificar etiquetas y clases

**Objetivo.** Asegurar la correspondencia entre etiquetas numéricas y nombres de clases. Las redes neuronales trabajan con índices, pero las predicciones deben interpretarse usando los nombres originales.

**Qué verificar:** que el índice de cada clase sea estable en entrenamiento, evaluación y futura aplicación. **Pregunta:** ¿qué error produciría intercambiar los índices de `humitas` y `pastel_de_choclo`?

El orden de las clases será importante posteriormente, porque deberá mantenerse durante el entrenamiento, la exportación del modelo y la aplicación final.


In [ ]:
for indice, clase in enumerate(class_names):
    print(indice, "→", clase)


## 16. Registrar las características del dataset

**Objetivo.** Registrar cómo se prepararon los datos para poder reproducir el trabajo y comunicar las decisiones técnicas.

**Qué hace el código:** genera una ficha con fuente, clases, número de imágenes, particiones, tamaño, normalización y transformaciones.

**Qué verificar:** que la ficha refleje los resultados realmente obtenidos. Para el proyecto integrador, complemente esta información con procedencia, permisos de uso y decisiones de limpieza.


In [ ]:
ficha_dataset = {
    "fuente": "Imágenes web de comida chilena (búsquedas en buscadores de imágenes)",
    "procedencia": "Ver fuentes_dataset_comida_chilena.csv (URL de la imagen y página de origen de cada archivo)",
    "limpieza": "Eliminación automática de duplicados y miniaturas (<150 px) + revisión manual de todas las imágenes",
    "permisos_de_uso": "Uso académico, sin redistribución: las imágenes pertenecen a sus autores",
    "numero_imagenes_validas": len(archivos_validos),
    "numero_clases": len(class_names),
    "clases": class_names,
    "distribucion_por_clase": df_resumen.set_index("Clase")["Cantidad"].to_dict(),
    "tamano_imagen": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "particion": {"train": len(X_train), "validation": len(X_val), "test": len(X_test)},
    "normalizacion": "0-1",
    "data_augmentation_solo_train": ["RandomFlip horizontal", "RandomRotation 0.10", "RandomZoom 0.10"],
    "archivos_problematicos": len(archivos_problematicos)
}
ficha_dataset


## 17. Actividad

**Propósito de la actividad.** Aplicar el procedimiento al dataset seleccionado para el proyecto integrador, no limitarse a ejecutar las celdas. Cada dupla debe documentar las diez características solicitadas y justificar sus decisiones.

**Evidencia esperada:** tablas de distribución, muestras visuales, diagnóstico de calidad, particiones y explicación de transformaciones. **Criterio de revisión:** que las conclusiones sean coherentes con las salidas efectivamente ejecutadas.

Se aplicó el notebook al dataset de comida chilena. Se registra:

1. Número total de imágenes.
2. Número de clases.
3. Cantidad de imágenes por clase.
4. Ejemplos representativos de cada categoría.
5. Dimensiones originales de las imágenes.
6. Archivos problemáticos detectados.
7. Estrategia de redimensionamiento y normalización.
8. Distribución definida para train, validation y test.
9. Existencia de desbalance entre clases.
10. Transformaciones de Data Augmentation seleccionadas y su justificación.


### 17.1 Evidencia calculada

La siguiente celda reúne en una tabla los datos de los puntos 1, 2, 3, 5, 6, 8 y 9, a partir de las salidas anteriores.

In [ ]:
resumen_por_clase = (
    df_imagenes.groupby("clase")
    .agg(imagenes=("archivo", "count"),
         ancho_min=("ancho", "min"), ancho_mediana=("ancho", "median"), ancho_max=("ancho", "max"),
         alto_min=("alto", "min"), alto_mediana=("alto", "median"), alto_max=("alto", "max"))
)
resumen_por_clase["porcentaje"] = (100 * resumen_por_clase["imagenes"] / resumen_por_clase["imagenes"].sum()).round(1)
resumen_por_clase["train"] = pd.Series(y_train).map(dict(enumerate(class_names))).value_counts()
resumen_por_clase["validation"] = pd.Series(y_val).map(dict(enumerate(class_names))).value_counts()
resumen_por_clase["test"] = pd.Series(y_test).map(dict(enumerate(class_names))).value_counts()

mayor, menor = resumen_por_clase["imagenes"].max(), resumen_por_clase["imagenes"].min()
print("1. Total de imágenes:", len(df_imagenes))
print("2. Número de clases:", len(class_names))
print("6. Archivos problemáticos:", len(archivos_problematicos))
print(f"8. Partición: train={len(X_train)}, validation={len(X_val)}, test={len(X_test)}")
print(f"9. Razón de desbalance (clase mayor / clase menor): {mayor / menor:.2f}")
assert not (set(X_train) & set(X_val) or set(X_train) & set(X_test) or set(X_val) & set(X_test)), "Hay rutas repetidas entre subconjuntos"
print("   Ninguna ruta se repite entre train, validation y test.")
resumen_por_clase


### 17.2 Respuestas

*Valores obtenidos con la versión del dataset de 1.074 imágenes. Si se agregan o eliminan fotos, deben actualizarse con las salidas de la celda 17.1.*

**1. Número total de imágenes.** 1.074 imágenes válidas, todas en formato JPEG.

**2. Número de clases.** 8 clases: `cazuela`, `chacarero`, `completo`, `empanada`, `humitas`, `mote_con_huesillo`, `pastel_de_choclo` y `sopaipilla`. Los índices numéricos siguen el orden alfabético de las carpetas (0 → cazuela … 7 → sopaipilla), como muestra la sección 15.

**3. Cantidad de imágenes por clase.**

| Clase | Imágenes | % |
|---|---|---|
| empanada | 155 | 14,4 |
| sopaipilla | 155 | 14,4 |
| cazuela | 154 | 14,3 |
| pastel_de_choclo | 141 | 13,1 |
| humitas | 132 | 12,3 |
| completo | 124 | 11,5 |
| chacarero | 121 | 11,3 |
| mote_con_huesillo | 92 | 8,6 |

**4. Ejemplos representativos de cada categoría.** La sección 6 muestra tres imágenes al azar por clase. Las fotografías corresponden a sus etiquetas, pero tienen fondos, encuadres e iluminación muy variados (fotos de recetas, de estudio y caseras). Hay pares de clases visualmente parecidos que podrían confundir al clasificador:
- **humitas y pastel de choclo**: ambos son preparaciones de choclo molido de color amarillo, aunque las humitas van envueltas en hojas de choclo y el pastel se sirve gratinado en una fuente de greda;
- **chacarero y completo**: ambos son sándwiches con pan y vegetales verdes encima (porotos verdes y ají en el chacarero; palta en el completo);
- **empanada y sopaipilla**: ambas son masas doradas.

**5. Dimensiones originales.** No son homogéneas. El ancho va de 250 a 6.000 px (mediana 1.024 px) y el alto, de 150 a 4.624 px (mediana 768 px). Hay imágenes horizontales, verticales y cuadradas, y todas están en modo RGB (3 canales).

**6. Archivos problemáticos.** `Image.verify()` no detectó archivos dañados o ilegibles (0). Antes de este notebook, durante la construcción del dataset, se eliminaron casi-duplicados (con *hash* perceptual), miniaturas de menos de 150 px e imágenes que no mostraban el plato terminado: ingredientes, preparación, afiches con texto, collages, personas u otros platos. Esa revisión se hizo de forma automática y también manual.

**7. Redimensionamiento y normalización.** Todas las imágenes se redimensionan a **224 × 224 px** con `tf.image.resize`, porque los *batches* requieren un tamaño fijo y 224 × 224 es un tamaño estándar para CNN. Esto deforma levemente las fotos que no son cuadradas, pero el plato sigue siendo reconocible. Los píxeles se escalan del rango 0–255 al rango **0–1** con `Rescaling(1./255)` en los tres subconjuntos, para que el modelo reciba entradas en la misma escala.

**8. Distribución train / validation / test.** División estratificada **70 / 15 / 15** con semilla 42: **751** imágenes de entrenamiento, **161** de validación y **162** de prueba. La estratificación mantiene la proporción de cada clase (por ejemplo, mote con huesillo: 64 / 14 / 14; empanada: 108 / 23 / 24), y se comprobó que ninguna ruta se repite entre subconjuntos. El conjunto *test* queda reservado para la evaluación final.

**9. Desbalance entre clases.** Hay un desbalance **moderado**: la clase mayor (empanada y sopaipilla, 155) tiene 1,68 veces las imágenes de la menor (mote con huesillo, 92). No es extremo, pero mote con huesillo podría tener un *recall* más bajo. Por eso conviene revisar las métricas por clase y no solo la *accuracy* global.

**10. Data Augmentation y justificación.** Se aplica solo al entrenamiento:
- **`RandomFlip("horizontal")`**: un plato fotografiado en espejo sigue siendo el mismo plato.
- **`RandomRotation(0.10)`** (unos ±36°): las fotos de comida se toman desde ángulos distintos y con el plato girado.
- **`RandomZoom(0.10)`**: simula fotos tomadas más cerca o más lejos.

No se usa el **reflejo vertical** ni rotaciones grandes, porque generarían imágenes poco realistas, como un vaso de mote con huesillo invertido. Tampoco se cambian los colores, porque el color es clave para distinguir platos (el amarillo del choclo, el verde de la palta o de los porotos verdes). Validation y test no reciben aumentos, para que la evaluación sea estable.



## 18. Cierre: dataset preparado para construir una CNN

**Cierre conceptual.** Preparar imágenes no equivale a entrenar una CNN. La calidad del conjunto, las etiquetas, la separación de datos y la consistencia del pipeline condicionan la validez de los experimentos posteriores.

**Autoevaluación:** ¿podría explicar por qué se excluyeron archivos dañados, se usó una división estratificada y se reservó test? ¿Puede señalar en qué etapa se aplica cada transformación?

El resultado debe ser un **dataset preparado, organizado y validado**, listo para la futura construcción y entrenamiento de una CNN.

**Dataset original → exploración → control de calidad → etiquetado → división estratificada → redimensionamiento → normalización → Data Augmentation solo en train → dataset preparado**.

Los pipelines finales son `train_aug`, `val_preparado` y `test_preparado`. No se entrena una CNN en este notebook.


## Lista de verificación — ¿Está mi dataset preparado para una CNN?

Antes de pasar al notebook de construcción y entrenamiento de la CNN, complete esta revisión usando las salidas obtenidas. Marque cada casilla **solo si puede justificarla con evidencia**:

- [ ] **1. Fuente y organización:** conozco la procedencia de las imágenes y las carpetas corresponden a clases identificables.
- [ ] **2. Exploración:** registré el total de imágenes, su distribución por clase, ejemplos visuales y dimensiones originales.
- [ ] **3. Calidad:** revisé archivos ilegibles y posibles errores de etiquetado; comprobé si existen duplicados o fotografías relacionadas que podrían causar fuga de datos.
- [ ] **4. Etiquetas:** verifiqué que cada índice numérico representa la clase correcta.
- [ ] **5. Partición:** train, validation y test son independientes y puedo explicar la proporción utilizada.
- [ ] **6. Estandarización:** las imágenes tienen tamaño y canales compatibles; los píxeles están normalizados de manera consistente.
- [ ] **7. Data Augmentation:** seleccioné transformaciones razonables y se aplican solo a train.
- [ ] **8. Pipeline:** identifiqué los datasets preparados (`train_aug`, `val_preparado`, `test_preparado`) y documenté las decisiones principales.

**Conclusión:** si una de estas condiciones no está resuelta, primero debe revisarse la preparación de los datos. **El próximo paso —en otro notebook— será diseñar, entrenar y evaluar la CNN utilizando estos conjuntos.**